In [ ]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io, os, pandas as pd

KEY_FILE = "/kaggle/input/datasets/kareemabosamra/api-key/speedy-cab-492712-h4-0fe1ced65a40.json"  # ← path to your key
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
creds = service_account.Credentials.from_service_account_file(KEY_FILE, scopes=SCOPES)
service = build("drive", "v3", credentials=creds)

FOLDER_ID = "1mgZiQCke45rUQ0HYYGyujvKJ9N94Vkt9"  # your folder ID

results = service.files().list(
    q=f"'{FOLDER_ID}' in parents and mimeType='text/csv'",
    fields="files(id, name)",
    pageSize=100
).execute()

files = results.get("files", [])
print(f"Found {len(files)} CSV files")

df_list = []
for file in files:
    print(f"  Reading: {file['name']}")
    request = service.files().get_media(fileId=file["id"])
    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buffer.seek(0)
    df_list.append(pd.read_csv(buffer))

combined = pd.concat(df_list, ignore_index=True)
print(f"\n✅ Total rows: {len(combined):,}")

combined.to_csv("/kaggle/working/combined_weather.csv", index=False)
print("Saved!")

Found 46 CSV files
  Reading: weather_batch_01950_01999.csv
  Reading: weather_batch_01850_01899.csv
  Reading: weather_batch_01800_01849.csv
  Reading: weather_batch_01900_01949.csv
  Reading: weather_batch_01750_01799.csv
  Reading: weather_batch_00000_00099.csv
  Reading: weather_batch_01700_01749.csv
  Reading: weather_batch_01650_01699.csv
  Reading: weather_batch_01600_01649.csv
  Reading: weather_batch_01550_01599.csv
  Reading: weather_batch_01500_01549.csv
  Reading: weather_batch_01450_01499.csv
  Reading: weather_batch_01400_01449.csv
  Reading: weather_batch_01350_01399.csv
  Reading: weather_batch_01300_01349.csv
  Reading: weather_batch_01250_01299.csv
  Reading: weather_batch_02300_02340.csv
  Reading: weather_batch_02250_02299.csv
  Reading: weather_batch_02200_02249.csv
  Reading: weather_batch_00950_00999.csv
  Reading: weather_batch_01200_01249.csv
  Reading: weather_batch_01150_01199.csv
  Reading: weather_batch_00900_00949.csv
  Reading: weather_batch_00850_00899.c

In [ ]:
import os
import json

os.makedirs("/kaggle/working/weather_dataset", exist_ok=True)

os.rename("/kaggle/working/combined_weather.csv", 
          "/kaggle/working/weather_dataset/combined_weather.csv")

metadata = {
    "title": "Combined Weather Dataset",
    "id": "kareemabosamra/combined-weather-dataset",  # ← change this
    "licenses": [{"name": "CC0-1.0"}]
}

with open("/kaggle/working/weather_dataset/dataset-metadata.json", "w") as f:
    json.dump(metadata, f)

!kaggle datasets create -p /kaggle/working/weather_dataset --public

Starting upload for file combined_weather.csv
100%|██████████████████████████████████████| 2.74G/2.74G [00:46<00:00, 63.5MB/s]
Upload successful: combined_weather.csv (3GB)
Your public Dataset is being created. Please check progress at https://www.kaggle.com/datasets/kareemabosamra/combined-weather-dataset
